In [36]:
import pandas as pd
from pycaret.classification import *

In [3]:
# Charger les fichiers CSV
df_large = pd.read_csv('../data/raw/bank-full.csv', sep=';')
df_small = pd.read_csv('../data/raw/bank.csv', sep=';')

## Nettoyage des données

### Petit Dataset

In [10]:
df_clean_small = df_small
df_clean_small = df_clean_small.rename(columns={"y":"subscribed", "default": "credit_in_default","contact":"contact_type", "pdays":"last_contact","previous":"number_of_contact_before_campaign","poutcome":"result_campaign" })

In [11]:
del df_clean_small["duration"]

In [13]:
df_clean_small = df_clean_small[df_clean_small['job'] != 'unknown']

In [14]:
# Création de la colonne contact_status

df_clean_small["contact_status"] = df_clean_small['last_contact'].apply(lambda x: 'no_contact' if x == -1 else 'already_contact')

In [16]:
df_clean_small.head()

,age,job,marital,education,credit_in_default,balance,housing,loan,contact_type,day,month,campaign,last_contact,number_of_contact_before_campaign,result_campaign,subscribed,contact_status
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,1,-1,0,unknown,no,no_contact
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,1,339,4,failure,no,already_contact
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,1,330,1,failure,no,already_contact
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,4,-1,0,unknown,no,no_contact
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact


### Grand Dataset

In [20]:
df_clean_large = df_large
df_clean_large = df_clean_large.rename(columns={"y":"subscribed", "default": "credit_in_default","contact":"contact_type", "pdays":"last_contact","previous":"number_of_contact_before_campaign","poutcome":"result_campaign" })

In [21]:
del df_clean_large["duration"]

In [22]:
df_clean_large = df_clean_large[df_clean_large['job'] != 'unknown']

In [23]:
df_clean_large["contact_status"] = df_clean_large['last_contact'].apply(lambda x: 'no_contact' if x == -1 else 'already_contact')

In [24]:
df_clean_large.head()

,age,job,marital,education,credit_in_default,balance,housing,loan,contact_type,day,month,campaign,last_contact,number_of_contact_before_campaign,result_campaign,subscribed,contact_status
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,-1,0,unknown,no,no_contact
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact
5,35,management,married,tertiary,no,231,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact


In [25]:
# Répartition des classes - Petit dataset
print("=== PETIT DATASET ===")
print("Nombre total d'observations:", len(df_clean_small))
print("\nRépartition brute:")
print(df_clean_small['subscribed'].value_counts())
print("\nRépartition en pourcentages:")
print(df_clean_small['subscribed'].value_counts(normalize=True) * 100)

print("\n" + "="*50)

# Répartition des classes - Grand dataset  
print("=== GRAND DATASET ===")
print("Nombre total d'observations:", len(df_clean_large))
print("\nRépartition brute:")
print(df_clean_large['subscribed'].value_counts())
print("\nRépartition en pourcentages:")
print(df_clean_large['subscribed'].value_counts(normalize=True) * 100)

=== PETIT DATASET ===
Nombre total d'observations: 4483

Répartition brute:
subscribed
no     3969
yes     514
Name: count, dtype: int64

Répartition en pourcentages:
subscribed
no     88.534464
yes    11.465536
Name: proportion, dtype: float64

=== GRAND DATASET ===
Nombre total d'observations: 44923

Répartition brute:
subscribed
no     39668
yes     5255
Name: count, dtype: int64

Répartition en pourcentages:
subscribed
no     88.302206
yes    11.697794
Name: proportion, dtype: float64


In [26]:
info_small = df_clean_small.info()
print(info_small)

<class 'pandas.core.frame.DataFrame'>
Index: 4483 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   age                                4483 non-null   int64 
 1   job                                4483 non-null   object
 2   marital                            4483 non-null   object
 3   education                          4483 non-null   object
 4   credit_in_default                  4483 non-null   object
 5   balance                            4483 non-null   int64 
 6   housing                            4483 non-null   object
 7   loan                               4483 non-null   object
 8   contact_type                       4483 non-null   object
 9   day                                4483 non-null   int64 
 10  month                              4483 non-null   object
 11  campaign                           4483 non-null   int64 
 12  last_contac

## Modélisation pour les critères de Campagne

### Selection des variables

In [35]:
variables_campagne = [
    'contact_type',
    'day',
    'month',
    'campaign',
    'last_contact',
    'number_of_contact_before_campaign',
    'result_campaign', 
    'subscribed',
    'contact_status']

# Création du dataset pour la modélisation (création d'une copie)
df_modelisation_small =  df_clean_small[variables_campagne].copy()

print("Dataset créé avec", len(variables_campagne)-1, "variables explicatives")
print("Taille du dataset:", df_modelisation_small.shape)

Dataset créé avec 8 variables explicatives
Taille du dataset: (4483, 9)


### Setup de Pycaret

In [41]:
# Environnement Pycaret

clf = setup(data=df_modelisation_small,
            target='subscribed',
            session_id=123,
           )


,Description,Value
0,Session id,123
1,Target,subscribed
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(4483, 9)"
5,Transformed data shape,"(4483, 25)"
6,Transformed train set shape,"(3138, 25)"
7,Transformed test set shape,"(1345, 25)"
8,Numeric features,4
9,Categorical features,4


## Analyse des résultats de configuration PyCaret

### Informations clés

* **Target :** `subscribed` (Binary : yes/no)
* **Mapping :** no=0, yes=1
* **Données originales :** 4,483 lignes, 9 colonnes
* **Données transformées :** 4,483 lignes, **25 colonnes**

### Transformation automatique des données

**Pourquoi 25 colonnes au lieu de 9 ?**

PyCaret a automatiquement appliqué un **one-hot encoding** des variables catégorielles :

* `contact_type` → contact_type_cellular, contact_type_telephone, etc.
* `month` → month_jan, month_feb, month_mar, etc.
* `result_campaign` → result_campaign_success, result_campaign_failure, etc.

### Division des données

* **Train (entraînement) :** 3,138 lignes (70%)
* **Test (évaluation) :** 1,345 lignes (30%)

Cette division permet d'entraîner les modèles sur 70% des données et de les évaluer sur les 30% restants pour mesurer leur performance sur des données non vues.

### Recherche du meilleur modèle

In [42]:
best_model = compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ada,Ada Boost Classifier,0.8910,0.7006,0.8910,0.8639,0.8599,0.1971,0.2537,0.1550
lr,Logistic Regression,0.8894,0.6956,0.8894,0.8628,0.8527,0.1459,0.2156,0.9060
ridge,Ridge Classifier,0.8885,0.6884,0.8885,0.8588,0.8558,0.1717,0.2282,0.0750
gbc,Gradient Boosting Classifier,0.8856,0.7089,0.8856,0.8550,0.8580,0.1961,0.2365,0.2150
dummy,Dummy Classifier,0.8853,0.5000,0.8853,0.7837,0.8314,0.0000,0.0000,0.0600
lda,Linear Discriminant Analysis,0.8843,0.6883,0.8843,0.8582,0.8625,0.2370,0.2670,0.0700
lightgbm,Light Gradient Boosting Machine,0.8830,0.6777,0.8830,0.8540,0.8596,0.2167,0.2463,0.1940
knn,K Neighbors Classifier,0.8779,0.6142,0.8779,0.8423,0.8502,0.1568,0.1866,0.1010
rf,Random Forest Classifier,0.8741,0.6712,0.8741,0.8454,0.8547,0.2048,0.2210,0.2320
et,Extra Trees Classifier,0.8700,0.6494,0.8700,0.8404,0.8510,0.1888,0.2010,0.1900
